# 第5・6回：データ可視化と機械学習入門 — タイタニックデータで学ぶ分類問題

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/Nakaura-T/DS_Seminar1_Public/blob/main/notebooks/session05_06_ml_classification.ipynb)

**DSゼミナールⅠ（2026年度）**  
熊本大学 データサイエンス学科

前回（第3・4回）ではPythonの基礎とPandas・Matplotlib・Seabornを学びました。今回は**タイタニック号の乗客データ**を使い、可視化から機械学習まで一気に体験します。「乗客が生き残ったかどうか」を予測する分類モデルを8種類作り、比較してみましょう。

---

## 📋 本日の目標

1. 今回使うライブラリ（Seaborn・scikit-learn）の役割と主な引数を説明できる
2. タイタニックデータを読み込み、欠損値を適切に処理できる
3. Matplotlib・Seabornで生存に関わるパターンを可視化できる
4. 機械学習（分類）の基本概念を理解できる
5. scikit-learnで8種類の分類器を実装・比較できる
6. 混同行列を読み取り、モデルの特性を考察できる

---

## 0. 準備：ライブラリの読み込みと概要

### 0-1. 今回使うライブラリ一覧


In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns

from sklearn.model_selection import train_test_split
from sklearn.metrics import accuracy_score, confusion_matrix, ConfusionMatrixDisplay
from sklearn.inspection import DecisionBoundaryDisplay

# 分類器 8 種
from sklearn.linear_model import LogisticRegression
from sklearn.tree import DecisionTreeClassifier
from sklearn.ensemble import RandomForestClassifier, GradientBoostingClassifier
from sklearn.neighbors import KNeighborsClassifier
from sklearn.svm import SVC
from sklearn.naive_bayes import GaussianNB
from sklearn.neural_network import MLPClassifier

sns.set_theme(style="whitegrid")
print("ライブラリの読み込み完了！")


### 0-2. 各ライブラリの役割

データ分析と機械学習では、目的に応じたライブラリを組み合わせて使います。今回登場するライブラリの役割を整理しておきましょう。

#### pandas（パンダス）— 表データの操作

第3・4回で学んだ通り、CSVや表形式のデータを扱う中心的なライブラリです。`DataFrame`（表）と`Series`（列）という2種類のデータ構造を提供します。


In [ ]:
# 復習：pandas の主な使い方
import pandas as pd

df = pd.read_csv("data.csv")   # 読み込み
df.head()                       # 先頭表示
df.describe()                   # 統計量
df["列名"].mean()               # 平均
df[df["age"] > 20]              # フィルタリング


#### NumPy（ナンパイ）— 数値計算の基盤

**NumPy（Numerical Python）** は、Pythonで数値計算を高速に行うためのライブラリです。`ndarray`（多次元配列）という強力なデータ構造を持ち、pandas・scikit-learnなど他のライブラリの内部でも使われています。


In [ ]:
import numpy as np

# ndarray（配列）の作成
arr = np.array([1, 2, 3, 4, 5])
print(arr * 2)        # [2 4 6 8 10]（全要素に一括演算）
print(arr.mean())     # 3.0
print(arr.std())      # 標準偏差

# 0〜1の乱数を生成（機械学習での初期化などに使う）
np.random.seed(42)
print(np.random.rand(3))   # [0.374 0.951 0.732]


> [!NOTE]
> pandasのDataFrameは内部的にNumPyの配列で動いています。`df.values` でDataFrameをNumPy配列に変換できます。scikit-learnに渡すデータも基本的にはNumPy配列（またはそれに準じる形式）です。


### 📝 演習 0-1

`import numpy as np` の後、以下を実行してNumPyの基本を確認してください。


In [ ]:
arr = np.array([10, 20, 30, 40, 50])
print("配列:", arr)
print("平均:", arr.mean())
print("標準偏差:", arr.std())
print("最大値:", arr.max())
print("2倍した配列:", arr * 2)


#### Seaborn（シーボーン）— 統計グラフの描画

**Seaborn** はMatplotlibの上に構築されたグラフライブラリで、統計的な可視化に特化しています。

| 特徴           | Matplotlib               | Seaborn                         |
| -------------- | ------------------------ | ------------------------------- |
| コードの量     | 多い（全部自分で設定）   | 少ない（デフォルトが美しい）    |
| 柔軟性         | 高い（細かく制御できる） | 中程度                          |
| 統計グラフ     | 自分で計算が必要         | `hue` などで自動分割            |
| 向いている場面 | 特殊・精密なグラフ       | 探索的分析（EDA）の素早い可視化 |


##### Seabornの代表的な関数

| 関数                 | 用途                                   |
| -------------------- | -------------------------------------- |
| `sns.histplot()`     | ヒストグラム（kde=True でKDE曲線付き） |
| `sns.scatterplot()`  | 散布図                                 |
| `sns.boxplot()`      | 箱ひげ図                               |
| `sns.barplot()`      | 棒グラフ（誤差棒付き）                 |
| `sns.heatmap()`      | ヒートマップ（相関行列などに）         |
| `sns.pairplot()`     | 全変数ペアの散布図                     |
| `sns.load_dataset()` | 練習用データセットを1行で読み込む      |

#### scikit-learn（サイキットラーン）— 機械学習の標準ライブラリ

**scikit-learn** は、Pythonで機械学習を行うためのデファクトスタンダードのライブラリです。分類・回帰・クラスタリング・前処理・モデル評価など、機械学習に必要なほぼすべての機能が揃っています。

```
scikit-learn の主な機能

sklearn.model_selection   → データ分割・交差検証
sklearn.preprocessing     → 正規化・エンコーディング
sklearn.linear_model      → 線形回帰・ロジスティック回帰
sklearn.tree              → 決定木
sklearn.ensemble          → ランダムフォレスト・勾配ブースティング
sklearn.neighbors         → k近傍法
sklearn.svm               → サポートベクターマシン
sklearn.naive_bayes       → ナイーブベイズ
sklearn.neural_network    → ニューラルネット（MLP）
sklearn.metrics           → 精度評価（accuracy・混同行列・F1スコアなど）
```

scikit-learnの最大の特長は、**全てのモデルが同じインターフェース**で使えることです。


In [ ]:
# どのモデルも必ず以下の3ステップで使える
model = SomeClassifier()       # ① モデルを作る
model.fit(X_train, y_train)    # ② 学習する
y_pred = model.predict(X_test) # ③ 予測する


### 0-3. Seaborn 基本練習（tipsデータ）

Titanicデータを扱う前に、Seabornの主要なグラフ関数を練習用データ（`tips`）で確認しておきましょう。各グラフの特徴・引数・使い方を体験してから本データの分析に進みます。

---

##### ① ヒストグラム（sns.histplot）

**ヒストグラム**は、数値データの「分布の形」を棒グラフで可視化するグラフです。  
横軸に値の範囲（ビン）、縦軸にその範囲に入るデータの件数を表します。  
`kde=True` を付けると、分布をなめらかな曲線（**KDE：カーネル密度推定**）で重ねて表示でき、全体の傾向が掴みやすくなります。  
`hue` を使うと、グループごとの分布の違いを一枚の図で比較できます。

> **使いどころ**: 「年齢・金額・得点などの数値が、どんな範囲に集中しているか」を把握したいとき

##### `sns.histplot()` の主な引数


In [ ]:
sns.histplot(
    data=tips,         # 使うDataFrame
    x='total_bill',    # X軸にする列名
    hue='sex',         # グループごとに色を分ける列
    bins=20,           # ヒストグラムのビン（棒）の数（多いほど細かく）
    kde=True,          # KDE曲線（なめらかな分布曲線）を重ねるか
    palette='Set2',    # グループの色をパレット名で指定
    alpha=0.7,         # 透明度（0=完全透明, 1=不透明）
    stat='count',      # Y軸の集計方法：'count'件数, 'density'密度, 'probability'確率
)


| 引数      | 説明                       | よく使う値                             |
| --------- | -------------------------- | -------------------------------------- |
| `data`    | 使うDataFrame              | `data=tips`                            |
| `x`       | X軸にする列名              | `x='total_bill'`                       |
| `hue`     | グループ分けする列         | `hue='sex'`                            |
| `bins`    | ビンの数（多いほど細かい） | `bins=20`                              |
| `kde`     | KDE曲線を重ねるか          | `kde=True`                             |
| `palette` | グループの色指定           | `palette='Set2'`                       |
| `alpha`   | 透明度（0〜1）             | `alpha=0.7`                            |
| `stat`    | Y軸の集計方法              | `'count'`（件数）, `'density'`（密度） |

##### 使用例：sns.histplot()


In [ ]:
# 練習用データ（tips）を読み込む — この節全体で使います
tips = sns.load_dataset("tips")
print(tips.head())


**tipsデータセットの列一覧**

| 列名         | 型       | 内容                     | 値の例                         |
| ------------ | -------- | ------------------------ | ------------------------------ |
| `total_bill` | float    | 食事代の合計金額（ドル） | 16.99, 10.34, 21.01 …          |
| `tip`        | float    | チップの金額（ドル）     | 1.01, 1.66, 3.50 …             |
| `sex`        | category | 支払いをした人の性別     | `Male` / `Female`              |
| `smoker`     | category | 喫煙席かどうか           | `Yes` / `No`                   |
| `day`        | category | 来店した曜日             | `Thur` / `Fri` / `Sat` / `Sun` |
| `time`       | category | 来店した時間帯           | `Lunch` / `Dinner`             |
| `size`       | int      | テーブルの人数           | 1〜6                           |

> このデータは、あるレストランで実際に記録された **244件**の食事情報です。  
> チップの金額や食事代に何が影響しているかを分析する練習に適しています。


In [ ]:
# 食事代（total_bill）の分布を性別（sex）で色分け
plt.figure(figsize=(8, 4))
sns.histplot(
    data=tips,
    x='total_bill',
    hue='sex',
    bins=20,
    kde=True,
    palette={'Male': '#4e79a7', 'Female': '#e15759'},
    alpha=0.7,
    stat='count'
)
plt.title("食事代の分布（性別別）", fontsize=13)
plt.xlabel("食事代（ドル）")
plt.ylabel("人数")
plt.tight_layout()
plt.show()


> **📝 練習問題（histplot）**  
> チップ（`tip`）の分布を、時間帯（`time`：Lunch / Dinner）で色分けしたヒストグラムを描いてください。  
> `bins=15, kde=True` を指定し、タイトルに「チップの分布（時間帯別）」と付けましょう。  
> ※ KDE曲線の有無でグラフの印象はどう変わりますか？

---

##### ② 散布図（sns.scatterplot）

**散布図**は、2つの数値変数の関係（相関）を点で可視化するグラフです。  
横軸・縦軸にそれぞれ数値列を指定し、1行のデータが1点として描かれます。  
`hue` でカテゴリ変数を色分けすることで、グループ間の違いや傾向の違いも同時に確認できます。  
右肩上がりに点が並ぶほど正の相関、右肩下がりなら負の相関があることを示します。

> **使いどころ**: 「2つの数値変数に関係があるか？」「グループによって傾向が違うか？」を確認したいとき

##### `sns.scatterplot()` の主な引数


In [ ]:
sns.scatterplot(
    data=tips,
    x='total_bill',    # X軸にする列名
    y='tip',           # Y軸にする列名
    hue='smoker',      # 点の色分けに使う列
    size='size',       # 点のサイズを変える列（省略可）
    palette='Set2',
    alpha=0.7,         # 透明度
    s=50,              # 点のサイズ（固定値で指定するとき）
)


| 引数      | 説明                 | よく使う値                |
| --------- | -------------------- | ------------------------- |
| `x`, `y`  | X軸・Y軸の列名       | `x='total_bill', y='tip'` |
| `hue`     | 点の色分けに使う列   | `hue='smoker'`            |
| `size`    | 点のサイズを変える列 | `size='size'`             |
| `alpha`   | 透明度               | `alpha=0.7`               |
| `s`       | 点のサイズ（固定）   | `s=50`                    |
| `palette` | カラーパレット       | `palette='Set2'`          |

##### 使用例：sns.scatterplot()


In [ ]:
# 食事代（total_bill）とチップ（tip）の関係を曜日（day）で色分け
plt.figure(figsize=(8, 5))
sns.scatterplot(
    data=tips,
    x='total_bill',
    y='tip',
    hue='day',
    palette='Set2',
    alpha=0.8,
    s=60
)
plt.title("食事代とチップの関係（曜日別）", fontsize=13)
plt.xlabel("食事代（ドル）")
plt.ylabel("チップ（ドル）")
plt.tight_layout()
plt.show()


> **📝 練習問題（scatterplot）**  
> 食事代（`total_bill`）とチップ（`tip`）の散布図を描き、色分けを「喫煙有無（`smoker`）」に変えてください。  
> `alpha=0.7, s=50` を指定しましょう。  
> ※ 喫煙席と禁煙席でチップの傾向に違いはありますか？グラフから読み取ってみましょう。

---

##### ③ 箱ひげ図（sns.boxplot）

**箱ひげ図**は、データの分布を「中央値・四分位数・外れ値」でまとめて表示するグラフです。

```
|--- ひげ（最小値〜第1四分位）---|  箱（第1〜第3四分位） |----- ひげ（第3四分位〜最大値）---|
                              ━━━━━━━|━━━━━━━
                                   中央値（第2四分位）
```

- **箱の上下端**：データの25%点（Q1）と75%点（Q3） → この幅を**IQR（四分位範囲）**という
- **箱の中の線**：中央値（50%点）
- **ひげの先端**：Q1 − 1.5×IQR ～ Q3 + 1.5×IQR の範囲
- **ひげの外の点**：外れ値

複数のグループを横に並べることで、グループ間の分布の違いを直感的に比較できます。

> **使いどころ**: 「グループごとに値のばらつきや中央値がどれだけ違うか」を比較したいとき

##### `sns.boxplot()` の主な引数


In [ ]:
sns.boxplot(
    data=tips,
    x='day',                           # X軸（カテゴリ列）
    y='total_bill',                    # Y軸（数値列）
    hue='smoker',                      # グループ分けする列
    palette='Set2',                    # カラーパレット
    order=['Thur', 'Fri', 'Sat', 'Sun'],  # X軸の順序を明示的に指定
    width=0.6,                         # 箱の幅
)


| 引数      | 説明              | よく使う値                         |
| --------- | ----------------- | ---------------------------------- |
| `x`       | X軸（カテゴリ列） | `x='day'`                          |
| `y`       | Y軸（数値列）     | `y='total_bill'`                   |
| `hue`     | グループ分け      | `hue='smoker'`                     |
| `palette` | カラーパレット    | `'Set2'`, `'Blues'`, `'viridis'`   |
| `order`   | X軸の順序         | `order=['Thur','Fri','Sat','Sun']` |

##### 使用例：sns.boxplot()


In [ ]:
# 曜日（day）ごとの食事代（total_bill）を喫煙有無（smoker）で色分け
plt.figure(figsize=(8, 5))
sns.boxplot(
    data=tips,
    x='day',
    y='total_bill',
    hue='smoker',
    palette={'Yes': '#e15759', 'No': '#4e79a7'},
    order=['Thur', 'Fri', 'Sat', 'Sun'],
    width=0.6
)
plt.title("曜日別の食事代（喫煙有無別）", fontsize=13)
plt.xlabel("曜日")
plt.ylabel("食事代（ドル）")
plt.tight_layout()
plt.show()


> **📝 練習問題（boxplot）**  
> 時間帯（`time`：Lunch / Dinner）ごとのチップ（`tip`）の箱ひげ図を描いてください。  
> `hue='sex'`（性別）で色分けし、`palette='Set2'` を使いましょう。  
> ※ ランチとディナーでチップの中央値はどちらが高いですか？外れ値はどちらに多いですか？

---

##### ④ ヒートマップ（sns.heatmap）

**ヒートマップ**は、2次元の表（行列）の値を色の濃淡で表現するグラフです。  
データサイエンスでは特に**相関行列**の可視化によく使われます。  
相関係数は −1〜+1 の値をとり、`cmap="coolwarm"` を使うと正の相関が赤・負の相関が青で直感的に把握できます。

| 相関係数の目安 | 意味             |
| -------------- | ---------------- |
| 0.7 以上       | 強い正の相関     |
| 0.4〜0.7       | 中程度の正の相関 |
| −0.4〜0.4      | ほぼ相関なし     |
| −0.7〜−0.4     | 中程度の負の相関 |
| −0.7 以下      | 強い負の相関     |

> **使いどころ**: 「どの変数とどの変数が強く関係しているか」を全ペアまとめて確認したいとき

##### `sns.heatmap()` の主な引数


In [ ]:
corr = tips.corr(numeric_only=True)
sns.heatmap(
    corr,              # 表示する2D配列（相関行列など）
    annot=True,        # 各セルに数値を表示するか
    fmt=".2f",         # 数値のフォーマット（小数2桁）
    cmap="coolwarm",   # カラーマップ（赤=正の相関、青=負の相関）
    vmin=-1, vmax=1,   # カラーの値の範囲
    linewidths=0.5,    # セル間の区切り線の太さ
    square=True,       # セルを正方形にするか
)


| 引数           | 説明               | よく使う値                        |
| -------------- | ------------------ | --------------------------------- |
| `annot`        | セルに数値を表示   | `annot=True`                      |
| `fmt`          | 数値のフォーマット | `".2f"`                           |
| `cmap`         | カラーマップ       | `"coolwarm"`, `"Blues"`, `"RdBu"` |
| `vmin`, `vmax` | カラーの範囲       | `vmin=-1, vmax=1`                 |
| `linewidths`   | セルの区切り線     | `linewidths=0.5`                  |
| `square`       | 正方形セルにするか | `square=True`                     |

##### 使用例：sns.heatmap()


In [ ]:
# tips の数値列の相関行列をヒートマップで表示
tips_corr = tips[['total_bill', 'tip', 'size']].corr()

plt.figure(figsize=(6, 5))
sns.heatmap(
    tips_corr,
    annot=True,
    fmt=".2f",
    cmap="coolwarm",
    vmin=-1, vmax=1,
    linewidths=0.5,
    square=True
)
plt.title("tipsデータの相関行列", fontsize=13)
plt.tight_layout()
plt.show()


> **📝 練習問題（heatmap）**  
> `tips` データ全体の数値列（`total_bill`, `tip`, `size`）の相関行列を計算し、ヒートマップを描いてください。  
> `cmap='Blues'` に変えて、見た目の違いを確認しましょう。  
> ※ `total_bill` と `tip` の相関係数はいくつですか？「チップは食事代に比例する」と言えそうですか？

---

### 📝 演習 0-2

`sns.get_dataset_names()` を実行して、Seabornに内蔵されているデータセットの一覧を確認してください。興味があるものを1つ `sns.load_dataset()` で読み込み `head()` を表示してみましょう。

### 📝 演習 0-3（Seaborn histplot）

`tips = sns.load_dataset("tips")` を読み込み、以下の2つのグラフを描いてください。

1. `total_bill`（食事代）のヒストグラムを `bins=20, kde=True` で描く
2. 同じグラフを `hue='sex'`（性別で色分け）で描き直す

引数 `kde=True` を `kde=False` に変えると何が変わりますか？

### 📝 演習 0-4（Seaborn scatterplot / boxplot）

`tips` データを使い、以下を描いてください。

1. `sns.scatterplot()` で X軸=`total_bill`、Y軸=`tip`、`hue='day'` の散布図
2. `sns.boxplot()` で X軸=`day`、Y軸=`total_bill`、`hue='sex'`、`palette='Set2'` の箱ひげ図

### 📝 演習 0-5（Seaborn heatmap）

`tips` の相関行列を計算し、`sns.heatmap()` で表示してください。  
`annot=True, fmt=".2f", cmap="coolwarm"` を指定し、`total_bill` と `tip` の相関係数を読み取ってください。


---

## 1. データ読み込みと確認

### タイタニック号について

1912年4月、イギリスの豪華客船タイタニック号は北大西洋で氷山に衝突し沈没しました。乗客・乗員2,224人のうち、生存者は約710人（約32%）でした。このデータセットには891人分の乗客情報と生死の記録が含まれており、機械学習の入門データとして世界中で使われています。


In [ ]:
df = sns.load_dataset("titanic")
df.head(10)


In [ ]:
print("データの形状:", df.shape)
print("\n列名の一覧:")
print(df.columns.tolist())


In [ ]:
print("\n各列の情報（型・欠損値の有無）:")
df.info()


In [ ]:
print("\n数値列の統計量:")
df.describe()


### タイタニックデータの主な列

| 列名       | 意味                                                           | 型    |
| ---------- | -------------------------------------------------------------- | ----- |
| `survived` | 生存(1) / 死亡(0)　← **目的変数**                              | int   |
| `pclass`   | 客室クラス（1=1等, 2=2等, 3=3等）                              | int   |
| `sex`      | 性別（male / female）                                          | str   |
| `age`      | 年齢                                                           | float |
| `sibsp`    | 同乗している兄弟・配偶者の人数 (Siblings and Spouses)          | int   |
| `parch`    | 同乗している親・子供の人数 (Parents and Children)              | int   |
| `fare`     | 運賃（ポンド）                                                 | float |
| `embarked` | 乗船港（C=シェルブール, Q=クイーンズタウン, S=サウサンプトン） | str   |

### 📝 演習 1-1

`df.isnull().sum()` を実行して、各列の欠損値の件数を確認してください。  
欠損値がある列はどれですか？欠損率（欠損数 ÷ 全件数 × 100）も計算してみましょう。


In [ ]:
# 欠損値の件数と欠損率を一覧表示する
missing = df.isnull().sum()
missing_rate = (missing / len(df) * 100).round(1)
pd.DataFrame({'欠損件数': missing, '欠損率(%)': missing_rate})[missing > 0]


### 📝 演習 1-2

`df["survived"].value_counts()` を実行して、生存者と死亡者の件数を確認してください。  
全体の生存率は何%ですか？（ヒント：`df["survived"].mean()` でも計算できます）

### 📝 演習 1-3

`df.describe()` の結果から、以下を読み取ってください。

1. 乗客の年齢（`age`）の平均・最大・最小はいくつですか？
2. 運賃（`fare`）の標準偏差は平均の何倍ですか？（ばらつきが大きいかどうかの目安になります）

---

## 2. 前処理

### 2-1. 欠損値とは何か

**欠損値（Missing Value）** とは、データの一部が記録されていない空欄のことです。現実のデータには必ずといっていいほど欠損値が含まれており、適切に処理しなければ機械学習モデルが動きません。

### 2-2. 欠損値の処理方法

欠損値の主な対処法は大きく2つです。

#### ① 削除（Deletion）


In [ ]:
# 欠損値がある行をすべて削除
df_dropped = df.dropna()
print("削除前:", len(df), "行")
print("削除後:", len(df_dropped), "行")


`dropna()` の主な引数：

| 引数     | 説明                                         | 例                            |
| -------- | -------------------------------------------- | ----------------------------- |
| `axis`   | 行(0)か列(1)を削除するか                     | `axis=0`（デフォルト=行削除） |
| `subset` | 指定列に欠損がある行だけ削除                 | `subset=['age']`              |
| `how`    | `'any'`（1つでも欠損）or `'all'`（全部欠損） | `how='any'`                   |

**メリット**：シンプルで実装が簡単  
**デメリット**：データが大量に失われる可能性がある（今回は `cabin` 列が77%欠損のため、削除するとほぼ全行が消える）

#### ② 補完（Imputation）

欠損値を何らかの値で埋める方法です。


In [ ]:
df_test = df[['age', 'fare']].copy()

# 数値列：平均値で補完
df_test['age_mean'] = df_test['age'].fillna(df_test['age'].mean())

# 数値列：中央値で補完（外れ値に強い）
df_test['age_median'] = df_test['age'].fillna(df_test['age'].median())

print("平均値補完後の欠損:", df_test['age_mean'].isnull().sum())
print("中央値補完後の欠損:", df_test['age_median'].isnull().sum())


`fillna()` の主な引数：

| 引数      | 説明                           | 例             |
| --------- | ------------------------------ | -------------- |
| `value`   | 埋める値（数値・文字列・辞書） | `fillna(28.0)` |
| `inplace` | 元のDataFrameを直接変更するか  | `inplace=True` |

**平均 vs 中央値：どちらで補完するか？**


In [ ]:
print(f"age の平均値: {df['age'].mean():.1f}")
print(f"age の中央値: {df['age'].median():.1f}")
print(f"age の歪度: {df['age'].skew():.2f}  （0に近いほど正規分布に近い）")

plt.figure(figsize=(8, 4))
sns.histplot(df['age'].dropna(), bins=30, kde=True, color='steelblue')
plt.axvline(df['age'].mean(), color='red', linestyle='--',
            label=f"平均: {df['age'].mean():.1f}")
plt.axvline(df['age'].median(), color='green', linestyle='--',
            label=f"中央値: {df['age'].median():.1f}")
plt.title("年齢の分布と補完値の比較")
plt.legend()
plt.show()


> [!NOTE]
> 分布が正規分布（左右対称）に近ければ平均値補完、右や左に裾が長い分布（偏り・外れ値がある）なら中央値補完が一般的に適しています。

### 2-3. 今回の前処理


In [ ]:
# ① 使う列だけ選択
cols = ['survived', 'pclass', 'sex', 'age', 'sibsp', 'parch', 'fare']
df_ml = df[cols].copy()

# ② 欠損値の処理（age の欠損を中央値で埋める）
print("処理前の欠損値:")
print(df_ml.isnull().sum())

df_ml['age'] = df_ml['age'].fillna(df_ml['age'].median())

print("\n処理後の欠損値:")
print(df_ml.isnull().sum())


In [ ]:
# ③ 文字列を数値に変換（sex: male → 0, female → 1）
# 機械学習モデルは文字列をそのまま扱えないため数値に変換する
df_ml['sex'] = df_ml['sex'].map({'male': 0, 'female': 1})

print("変換後の先頭5行:")
df_ml.head()


In [ ]:
# ④ 説明変数(X)と目的変数(y)に分ける
X = df_ml.drop('survived', axis=1)
y = df_ml['survived']

print("説明変数 X の形状:", X.shape)
print("目的変数 y の形状:", y.shape)
print("\n説明変数の列:", X.columns.tolist())
print("目的変数の値の種類:", y.unique())


> [!NOTE]
> **説明変数（X）**：モデルへの「入力」となる特徴量（年齢・性別・運賃など）  
> **目的変数（y）**：モデルが予測したい「答え」（生存=1 / 死亡=0）  
> 変数名に `X`（大文字）と `y`（小文字）を使うのはデータサイエンスの慣習です。

### 📝 演習 2-1

`age` 列を平均値で補完した場合と中央値で補完した場合で、それぞれヒストグラムを描いて比較してください。元の分布（欠損値を除いた `age`）と比べてどちらが形に近いですか？

### 📝 演習 2-2

`df_ml['sex'].value_counts()` を実行して、男女の人数を確認してください。  
また `df_ml.dtypes` を実行して、全列が数値型になっていることを確認してください。

### 📝 演習 2-3

`df_ml.describe()` を実行してください。`fare`（運賃）の最大値と最小値の差（範囲）を計算してください。また `age` の範囲も計算し、2つの特徴量のスケールの違いを確認してください。

---

## 3. 可視化演習

前処理が完了したデータを使って、生存に関わるパターンを視覚的に探ります。  
グラフから「どんな人が生き残りやすかったか」を読み取ってみましょう。


### 性別と生存率


In [ ]:
survival_by_sex = df_ml.groupby('sex')['survived'].mean()
survival_by_sex.index = ['男性(0)', '女性(1)']

plt.figure(figsize=(6, 4))
survival_by_sex.plot.bar(color=['#4e79a7', '#e15759'], rot=0)
plt.title("性別ごとの生存率", fontsize=14)
plt.xlabel("性別")
plt.ylabel("生存率")
plt.ylim(0, 1)
plt.tight_layout()
plt.show()
print(survival_by_sex)


### 年齢の分布（生存/死亡別）


In [ ]:
plt.figure(figsize=(10, 5))
sns.histplot(
    data=df_ml,
    x='age',
    hue='survived',
    bins=30,
    palette={0: '#e15759', 1: '#4e79a7'},
    alpha=0.7,
    kde=True        # KDE曲線を重ねて分布の形を把握しやすくする
)
plt.title("年齢分布（生存/死亡別）", fontsize=14)
plt.xlabel("年齢")
plt.ylabel("人数")
plt.legend(title="生存", labels=["死亡(0)", "生存(1)"])
plt.tight_layout()
plt.show()


### 散布図：年齢 × 運賃（生存/死亡の色分け）


In [ ]:
plt.figure(figsize=(10, 6))
sns.scatterplot(
    data=df_ml,
    x='age',
    y='fare',
    hue='survived',
    palette={0: '#e15759', 1: '#4e79a7'},
    alpha=0.6,
    s=40
)
plt.title("年齢 × 運賃（生存/死亡の分布）", fontsize=14)
plt.xlabel("年齢")
plt.ylabel("運賃（ポンド）")
plt.tight_layout()
plt.show()


### 📝 演習 3-5

客室クラス（`pclass`）ごとの生存率を `groupby().mean()` で計算し、棒グラフで表示してください。  
タイトルと軸ラベルを必ず付けてください。1等・2等・3等で差はありますか？

### 📝 演習 3-6

`sns.boxplot()` を使い、以下のグラフを描いてください。


In [ ]:
plt.figure(figsize=(8, 5))
sns.boxplot(
    data=df_ml,
    x='pclass',        # X軸：客室クラス
    y='fare',          # Y軸：運賃
    palette='Blues',   # カラーパレット
    order=[1, 2, 3]    # X軸の表示順を明示
)
plt.title("客室クラスごとの運賃分布", fontsize=14)
plt.xlabel("客室クラス")
plt.ylabel("運賃（ポンド）")
plt.tight_layout()
plt.show()


さらに `hue='survived'` を追加して、生存/死亡でも分けたグラフを描いてください。

### 📝 演習 3-7

`df_ml.corr()` で相関行列を計算し、`sns.heatmap()` で表示してください。  
以下の引数を必ず使ってください。


In [ ]:
corr = df_ml.corr(numeric_only=True)
plt.figure(figsize=(8, 6))
sns.heatmap(
    corr,
    annot=True,
    fmt=".2f",
    cmap="coolwarm",
    vmin=-1, vmax=1,
    square=True,
    linewidths=0.5
)
plt.title("特徴量の相関行列", fontsize=14)
plt.tight_layout()
plt.show()


`survived`（生存）と最も相関が高い特徴量はどれですか？その値はいくつですか？

### 📝 演習 3-8

`sns.barplot()` を使い、`sex`（性別）と `pclass`（客室クラス）の組み合わせ別の生存率を表示してください。


In [ ]:
# 元のdfを使う（sex が male/female のままの方が見やすい）
plt.figure(figsize=(8, 5))
sns.barplot(
    data=df,
    x='pclass',
    y='survived',
    hue='sex',
    palette={'male': '#4e79a7', 'female': '#e15759'}
)
plt.title("客室クラス × 性別の生存率", fontsize=14)
plt.xlabel("客室クラス")
plt.ylabel("生存率")
plt.tight_layout()
plt.show()


どのグループの生存率が最も高いですか？最も低いですか？

### 📝 演習 3-9（応用）

`df_ml['family_size'] = df_ml['sibsp'] + df_ml['parch'] + 1` で家族人数の列を作り、  
家族人数ごとの生存率を棒グラフで表示してください。一人旅（family_size=1）と家族連れで差はありますか？

---

## 4. 機械学習とは

### 4-1. 機械学習の定義

**機械学習（Machine Learning）** とは、データの中にある**パターンをコンピュータに自動的に学習させる**技術です。

人間がルールを明示的にプログラムする代わりに、大量のデータからコンピュータ自身がルールを発見します。

```
【従来のプログラミング】
  ルール（人間が作成） + データ  →  出力

【機械学習】
  データ + 正解ラベル  →  ルール（モデル）を自動で学習  →  新しいデータへの予測
```

**今回の例：** 年齢・性別・運賃などのデータと「実際の生死（正解）」を大量に与えることで、「こういう特徴の人は生き残る可能性が高い」というルールをモデルが自動的に発見します。

### 4-2. 機械学習の種類

| 種類             | 概要                             | 例                                 |
| ---------------- | -------------------------------- | ---------------------------------- |
| **教師あり学習** | 正解ラベルありのデータで学習する | 生存予測・住宅価格予測・スパム検出 |
| **教師なし学習** | 正解なしでデータの構造を発見する | クラスタリング・異常検知・次元削減 |
| **強化学習**     | 試行錯誤で最適な行動を学習する   | ゲームAI・ロボット制御・自動運転   |

今回は**教師あり学習**を扱います。

### 4-3. 分類と回帰

教師あり学習は、答えの種類によってさらに2つに分かれます。

| タスク                     | 答えの形                     | 評価指標                 | 例               |
| -------------------------- | ---------------------------- | ------------------------ | ---------------- |
| **分類（Classification）** | カテゴリ（生存/死亡、猫/犬） | 正解率・F1スコアなど     | **今回はこちら** |
| **回帰（Regression）**     | 連続値（価格・気温・点数）   | 平均二乗誤差（RMSE）など | 住宅価格予測     |

### 4-4. モデルとは何か

**モデル** とは、入力（特徴量）から出力（予測）を導き出す「関数」です。  
学習とは、訓練データに最もよく合うようにこの関数のパラメータを調整するプロセスです。

```
      特徴量 X              モデル（関数）               予測 ŷ
  ─────────────────     ──────────────────────      ─────────────
  age=22                                            生存確率: 0.75
  sex=1（女性）    →    f(X) = ... （学習済み）  →   → 予測: 生存(1)
  pclass=1
  fare=71.3
```

### 4-5. 訓練データとテストデータ

機械学習では、データを**訓練データ（学習用）** と **テストデータ（評価用）** に必ず分けます。

```
全データ（891件）
    ├── 訓練データ（712件 = 80%）：モデルの学習に使う
    └── テストデータ（179件 = 20%）：学習には使わず、精度評価だけに使う
```

**なぜ分けるのか？**

学習に使ったデータで精度を測ると「答えを見てテストする」ことになり、実際の予測能力が分かりません。テストデータを「初めて見るデータ」として扱うことで、モデルの本当の汎化性能を正しく評価できます。


In [ ]:
X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42, stratify=y
)

print(f"訓練データ: {X_train.shape[0]} 件")
print(f"テストデータ: {X_test.shape[0]} 件")
print(f"訓練データの生存率: {y_train.mean():.3f}")
print(f"テストデータの生存率: {y_test.mean():.3f}")


> [!NOTE]
> `random_state=42` は「乱数のシード」です。同じ値を指定することで、何度実行しても同じ分割結果が得られます。これにより実験の再現性が保たれます。

### 4-6. 過学習と汎化

機械学習で最も重要な概念の一つが**過学習（Overfitting）** です。

```
過学習とは：
  訓練データに特化しすぎて、新しいデータへの予測精度が下がる現象

「教科書の問題を丸暗記して、試験の応用問題が解けない」状態

  訓練データ精度：99%（丸暗記）
  テストデータ精度：65%（応用できない）← 過学習
```

**汎化（Generalization）** とは、学習していない新しいデータにも正しく予測できる能力のことです。機械学習の最終目標は「訓練データで高精度」ではなく「テストデータで高精度」です。


In [ ]:
# 過学習の例：max_depth を極端に大きくした決定木
for depth in [2, 5, 10, 20, None]:
    dt = DecisionTreeClassifier(max_depth=depth, random_state=42)
    dt.fit(X_train, y_train)
    train_acc = accuracy_score(y_train, dt.predict(X_train))
    test_acc  = accuracy_score(y_test,  dt.predict(X_test))
    label = str(depth) if depth else "無制限"
    print(f"max_depth={label:5s}  訓練: {train_acc:.3f}  テスト: {test_acc:.3f}")


### 4-7. 正解率（Accuracy）とは

```
正解率 = 正しく予測できた件数 ÷ テストデータの全件数

例：179件のテストデータで150件正解 → Accuracy = 150/179 ≈ 0.838（83.8%）
```


In [ ]:
from sklearn.metrics import accuracy_score

y_pred_example = [1, 0, 1, 1, 0]   # モデルの予測
y_true_example = [1, 0, 0, 1, 0]   # 実際の答え

acc = accuracy_score(y_true_example, y_pred_example)
print(f"正解率: {acc:.3f}")   # 4/5 = 0.800


### 📝 演習 4-1

`train_test_split` の `test_size` を 0.1・0.2・0.3 に変えて、それぞれの訓練/テストの件数を確認してください。  
`test_size` を大きくするとモデルの評価にどんな影響があると思いますか？

### 📝 演習 4-2

上の「過学習の例」コードを実行して、`max_depth` が大きくなるにつれ訓練精度とテスト精度の差がどう変化するか確認してください。`max_depth=None` のとき訓練精度は何%になりましたか？これは何を意味しますか？

### 4-8. scikit-learn の使い方：データ分割と学習の流れ

機械学習の概念を理解したところで、次は実際の手順を確認します。scikit-learnでは「データ分割 → 学習 → 予測 → 評価」という流れが標準パターンです。ここでは特に**データ分割**と**学習・予測の基本操作**を押さえましょう。

##### `train_test_split()` の主な引数


In [ ]:
X_train, X_test, y_train, y_test = train_test_split(
    X, y,              # 分割するデータ（説明変数と目的変数）
    test_size=0.2,     # テストデータの割合（0〜1）または件数
    random_state=42,   # 乱数シード（同じ値なら毎回同じ分割）
    stratify=y,        # 層化サンプリング（クラスの比率を維持する）
    shuffle=True,      # 分割前にシャッフルするか（デフォルトTrue）
)


| 引数           | 説明                                 | よく使う値                   |
| -------------- | ------------------------------------ | ---------------------------- |
| `test_size`    | テストの割合（0〜1）または件数       | `0.2`（20%）                 |
| `random_state` | 乱数シード（再現性のため）           | `42`（慣習的によく使われる） |
| `stratify`     | 層化サンプリング（クラス比率を保つ） | `stratify=y`                 |
| `shuffle`      | 分割前にシャッフルするか             | `True`（デフォルト）         |

> [!NOTE]
> **stratify とは？** 例えば生存率が38%のデータを分割するとき、`stratify=y` を指定すると訓練データ・テストデータ双方の生存率が38%付近になるよう調整されます。クラスの偏りが大きいデータで特に重要です。

### 📝 演習 4-3（sklearn train_test_split）

以下を試して、`stratify` の効果を確認してください。


In [ ]:
# stratify なし
X_tr1, X_te1, y_tr1, y_te1 = train_test_split(X, y, test_size=0.2, random_state=42)

# stratify あり
X_tr2, X_te2, y_tr2, y_te2 = train_test_split(X, y, test_size=0.2, random_state=42, stratify=y)

print("全体の生存率:               ", y.mean().round(3))
print("stratify なし - 訓練:", y_tr1.mean().round(3), " テスト:", y_te1.mean().round(3))
print("stratify あり - 訓練:", y_tr2.mean().round(3), " テスト:", y_te2.mean().round(3))


`stratify=y` を指定したときの方が、訓練/テストの生存率が全体に近くなっていますか？

### 📝 演習 4-4（sklearn fit / predict）

以下のコードを実行して、scikit-learnの基本パターンを確認してください。


In [ ]:
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import accuracy_score

# ① モデルを作る
lr = LogisticRegression(max_iter=1000)

# ② 学習する
lr.fit(X_train, y_train)
print("学習完了")

# ③ 予測する
y_pred_lr = lr.predict(X_test)
print("予測結果（先頭10件）:", y_pred_lr[:10])
print("実際の値（先頭10件）:", y_test.values[:10])

# ④ 正解率を計算する
acc = accuracy_score(y_test, y_pred_lr)
print(f"\n正解率: {acc:.4f}  ({acc*100:.1f}%)")


`predict_proba()` を使うと予測確率も取得できます。


In [ ]:
proba = lr.predict_proba(X_test)
print("予測確率（先頭5件）:")
print("  死亡確率   生存確率")
for i, p in enumerate(proba[:5]):
    print(f"  {p[0]:.3f}      {p[1]:.3f}  → 予測: {y_pred_lr[i]}")


---

## 5. 分類器の紹介

scikit-learnには多数の分類器が用意されています。今回は代表的な8種類を使います。どれが「最良」というわけではなく、データによって向き不向きがあります。現実的な分析では複数を試して精度を比較するのが主流です。

### 8つの分類器と主な引数

#### 1. LogisticRegression — 直線で空間を二分する

直線（超平面）1枚で空間を二分する最もシンプルな分類器です。解釈しやすく高速で、ベースラインとして最初に試すモデルとして適しています。


In [ ]:
LogisticRegression(
    max_iter=1000,    # 最大反復回数（デフォルト100では収束しないことがある）
    C=1.0,            # 正則化の強さ（小さいほど強い正則化→シンプルなモデル）
    random_state=42,
)


| 引数       | 説明                                   | よく使う値          |
| ---------- | -------------------------------------- | ------------------- |
| `max_iter` | 最大反復回数（不足するとWarning）      | `1000`              |
| `C`        | 正則化パラメータ（大きいほど正則化弱） | `1.0`（デフォルト） |

#### 2. DecisionTreeClassifier — Yes/No質問の木構造

「年齢が30以上か？」のようなYes/No質問を繰り返す木構造で分類します。結果が人間に説明しやすい（解釈可能性が高い）のが特長です。


In [ ]:
DecisionTreeClassifier(
    max_depth=5,           # 木の最大の深さ（大きすぎると過学習）
    min_samples_split=2,   # 分岐に必要な最小サンプル数
    min_samples_leaf=1,    # 葉ノードに必要な最小サンプル数
    criterion='gini',      # 分岐の評価基準（'gini' または 'entropy'）
    random_state=42,
)


| 引数               | 説明                                     | よく使う値                          |
| ------------------ | ---------------------------------------- | ----------------------------------- |
| `max_depth`        | 木の深さの上限（`None`で制限なし）       | `3`〜`10`                           |
| `criterion`        | 分岐の評価基準                           | `'gini'`（デフォルト）, `'entropy'` |
| `min_samples_leaf` | 葉の最小サンプル数（増やすと過学習抑制） | `1`〜`5`                            |

#### 3. RandomForestClassifier — 大量の木の多数決

Decision Treeを大量に作り多数決をとるアンサンブル手法です。高精度で安定しており、汎用的に使えます。


In [ ]:
RandomForestClassifier(
    n_estimators=100,  # 作る木の本数（多いほど安定するが遅くなる）
    max_depth=None,    # 各木の深さ制限
    max_features='sqrt', # 各分岐で使う特徴量の数（'sqrt'=√特徴量数）
    n_jobs=-1,         # 並列処理（-1で全CPUを使用）
    random_state=42,
)


| 引数           | 説明                     | よく使う値             |
| -------------- | ------------------------ | ---------------------- |
| `n_estimators` | 木の本数                 | `100`〜`500`           |
| `max_features` | 各分岐で考慮する特徴量数 | `'sqrt'`（デフォルト） |
| `n_jobs`       | 並列処理数               | `-1`（全CPU使用）      |

#### 4. GradientBoostingClassifier — 弱い木を積み重ねて改善

弱い木を順番に作り、前の木の間違いを少しずつ修正していくアンサンブル手法です。高精度でKaggleコンテストでも人気です。


In [ ]:
GradientBoostingClassifier(
    n_estimators=100,    # 作る木の本数
    learning_rate=0.1,   # 学習率（小さいほど慎重に学習）
    max_depth=3,         # 各木の深さ
    random_state=42,
)


| 引数            | 説明                     | よく使う値    |
| --------------- | ------------------------ | ------------- |
| `n_estimators`  | 木の本数                 | `100`〜`300`  |
| `learning_rate` | 学習率（小さいほど慎重） | `0.05`〜`0.2` |
| `max_depth`     | 各木の深さ               | `3`〜`5`      |

#### 5. KNeighborsClassifier — 近傍点の多数決

新しいデータに最も近いk件の訓練データの多数決で分類します。直感的に理解しやすい手法です。


In [ ]:
KNeighborsClassifier(
    n_neighbors=5,       # 参照する近傍点の数k
    metric='euclidean',  # 距離の計算方法
    weights='uniform',   # 近傍点の重み（'uniform'=均等, 'distance'=距離で重み付け）
)


| 引数          | 説明                              | よく使う値                        |
| ------------- | --------------------------------- | --------------------------------- |
| `n_neighbors` | 近傍点の数k（小さすぎると過学習） | `3`〜`15`                         |
| `metric`      | 距離計算方法                      | `'euclidean'`（ユークリッド距離） |
| `weights`     | 近傍点の重み                      | `'uniform'`, `'distance'`         |

#### 6. SVC — マージン最大の境界を探す

クラス間のマージン（余白）が最大になる境界を探す手法です。`kernel` パラメータで直線から曲線まで柔軟に対応できます。


In [ ]:
SVC(
    kernel='rbf',    # カーネルの種類（'linear','rbf','poly','sigmoid'）
    C=1.0,           # 正則化パラメータ
    gamma='scale',   # RBFカーネルのパラメータ
    probability=False, # 確率を出力するか（Trueにするとpredict_probaが使える）
)


| 引数     | 説明             | よく使う値                        |
| -------- | ---------------- | --------------------------------- |
| `kernel` | カーネルの種類   | `'rbf'`（デフォルト）, `'linear'` |
| `C`      | 正則化パラメータ | `0.1`〜`10`                       |
| `gamma`  | RBFカーネルの幅  | `'scale'`（デフォルト）           |

#### 7. GaussianNB — 確率の積で判定

ベイズの定理を使った確率的分類器です。各特徴量が正規分布に従いかつ独立と仮定します（Naive=素朴な仮定）。計算が高速で、テキスト分類やスパムフィルタによく使われます。


In [ ]:
GaussianNB(
    var_smoothing=1e-9,  # 数値安定性のための平滑化パラメータ
)


#### 8. MLPClassifier — 神経回路を模した多層構造

脳の神経回路を模した多層ネットワークで複雑なパターンを学習します。隠れ層のサイズと数で表現力を調整できます。


In [ ]:
MLPClassifier(
    hidden_layer_sizes=(100, 50),  # 隠れ層の構成（タプルで各層のノード数）
    activation='relu',             # 活性化関数（'relu','tanh','logistic'）
    max_iter=1000,                 # 最大反復回数
    learning_rate_init=0.001,      # 学習率
    random_state=42,
)


| 引数                 | 説明         | よく使う値             |
| -------------------- | ------------ | ---------------------- |
| `hidden_layer_sizes` | 隠れ層の構成 | `(100,)`, `(100, 50)`  |
| `activation`         | 活性化関数   | `'relu'`（デフォルト） |
| `max_iter`           | 最大反復回数 | `1000`                 |

### 各分類器の決定境界を可視化する

2つの特徴量（Age × Fare）だけを使ったとき、各分類器がどんな**決定境界**（分類の境い目）を引くかを可視化します。


In [ ]:
feature_x, feature_y = 'age', 'fare'
X_vis = df_ml[[feature_x, feature_y]].values
y_vis = df_ml['survived'].values

X_vtr, X_vte, y_vtr, y_vte = train_test_split(
    X_vis, y_vis, test_size=0.2, random_state=42
)

models_vis = {
    "Logistic Regression" : LogisticRegression(max_iter=1000),
    "Decision Tree"       : DecisionTreeClassifier(max_depth=4, random_state=42),
    "Random Forest"       : RandomForestClassifier(n_estimators=100, random_state=42),
    "Gradient Boosting"   : GradientBoostingClassifier(random_state=42),
    "KNN (k=5)"           : KNeighborsClassifier(n_neighbors=5),
    "SVM (RBF)"           : SVC(kernel='rbf'),
    "Naive Bayes"         : GaussianNB(),
    "Neural Net"          : MLPClassifier(hidden_layer_sizes=(50, 50), max_iter=1000, random_state=42),
}

fig, axes = plt.subplots(2, 4, figsize=(18, 9))
axes = axes.flatten()

for ax, (name, model) in zip(axes, models_vis.items()):
    model.fit(X_vtr, y_vtr)
    acc = accuracy_score(y_vte, model.predict(X_vte))

    DecisionBoundaryDisplay.from_estimator(
        model, X_vtr, ax=ax,
        response_method="predict",
        cmap="RdBu", alpha=0.55
    )
    ax.scatter(X_vtr[:, 0], X_vtr[:, 1],
               c=y_vtr, cmap="RdBu",
               edgecolors='white', linewidths=0.5, s=20, alpha=0.85)
    ax.set_title(f"{name}\n精度（2特徴量）: {acc:.2f}", fontsize=10, fontweight='bold')
    ax.set_xlabel("Age（年齢）", fontsize=9)
    ax.set_ylabel("Fare（運賃）", fontsize=9)

plt.suptitle("各分類器の決定境界（Age × Fare の2特徴量）\n  青=生存(1)  赤=死亡(0)",
             fontsize=13, fontweight='bold')
plt.tight_layout()
plt.show()


**グラフの見方：**

- **Logistic Regression**：きれいな直線1本。シンプルだが直線で分けられないパターンには弱い
- **Decision Tree**：カクカクした軸平行の境界。「もし〜なら」の条件が境界の形に現れる
- **Random Forest / Gradient Boosting**：Treeよりも滑らかで複雑な境界
- **KNN**：データ点に張り付くような不規則な境界。k が小さいほど複雑（過学習しやすい）
- **SVM (RBF)**：曲線を含むが全体的にシャープな境界
- **Naive Bayes**：楕円形の確率分布に基づくなめらかな境界
- **Neural Net**：最も複雑で柔軟な境界

> [!NOTE]
> この可視化は「2特徴量だけ」での結果です。次のセクションでは全特徴量（pclass・sex・age・sibsp・parch・fare）を使って、より高精度な学習を行います。

### 8分類器のメリット・デメリットまとめ

| 分類器 | メリット | デメリット | 向いている場面 |
| --- | --- | --- | --- |
| **Logistic Regression** | 高速・結果の解釈が容易（係数＝各特徴量の影響度）。確率値を直接出力できる | 直線（超平面）でしか分離できず、複雑なパターンに弱い | ベースライン構築、医療など解釈性が重要な場面 |
| **Decision Tree** | 結果を「もし〜なら」で人間に説明できる。前処理（標準化等）が不要 | 過学習しやすい。データの小さな変化で木の形が大きく変わる（不安定） | 意思決定の根拠を説明する必要がある場面 |
| **Random Forest** | 高精度で安定。過学習しにくく、特徴量重要度も算出できる | 計算コストが大きい。モデル内部の解釈が困難 | 汎用的な分類・回帰タスク全般。最初に試すモデルとして優秀 |
| **Gradient Boosting** | 精度が最も高くなることが多い。Kaggle等のコンペで上位常連 | 学習が遅い。ハイパーパラメータの調整が難しい | 精度を最大限追求したい場面（XGBoost・LightGBMとして実務で多用） |
| **KNN (k近傍法)** | 仕組みが直感的。学習フェーズがほぼ不要 | 予測が遅い（全データとの距離計算が必要）。高次元データに弱い（次元の呪い） | 小規模データ、推薦システム |
| **SVM** | 高次元データに強い。カーネル関数で柔軟な境界を描ける | 大規模データでは計算が非常に遅い。パラメータ調整（C・gamma）が重要 | テキスト分類、画像認識（特徴量が多くサンプルが少ない場面） |
| **Naive Bayes** | 極めて高速。少量データでもそこそこの精度が出る | 特徴量間の独立性を仮定するため、相関が強い特徴量があると精度低下 | テキスト分類（スパムフィルタ等）、リアルタイム予測 |
| **Neural Net (MLP)** | 複雑な非線形パターンを学習可能。柔軟性が最も高い | 大量データと計算資源が必要。ブラックボックスで解釈困難。過学習しやすい | 大規模データ、画像・音声・自然言語処理（深層学習の基礎） |

> [!TIP]
> 実務では**まず Logistic Regression をベースライン**として試し、次に **Random Forest や Gradient Boosting** で精度向上を狙うのが定石です。「最良のモデル」はデータによって異なるため、複数のモデルを比較することが重要です。

### 📝 演習 5-1

決定境界のグラフを見て、以下を答えてください。

1. 直線の境界を引いているモデルはどれですか？
2. 最も複雑（ギザギザ）な境界を引いているモデルはどれですか？
3. 複雑な境界が必ずしも「良いモデル」とは言えない理由を、過学習の概念と結びつけて考えてみましょう。

### 📝 演習 5-2（引数の効果を確認）

`KNeighborsClassifier` の `n_neighbors` を 1・3・5・15 に変えて、決定境界の形がどう変わるか確認してください。


In [ ]:
fig, axes = plt.subplots(1, 4, figsize=(18, 4))

for ax, k in zip(axes, [1, 3, 5, 15]):
    knn = KNeighborsClassifier(n_neighbors=k)
    knn.fit(X_vtr, y_vtr)
    acc = accuracy_score(y_vte, knn.predict(X_vte))

    DecisionBoundaryDisplay.from_estimator(knn, X_vtr, ax=ax, cmap="RdBu", alpha=0.5)
    ax.scatter(X_vtr[:, 0], X_vtr[:, 1], c=y_vtr, cmap="RdBu",
               edgecolors='white', linewidths=0.5, s=15, alpha=0.8)
    ax.set_title(f"k={k}  精度: {acc:.2f}", fontsize=11, fontweight='bold')
    ax.set_xlabel("Age"); ax.set_ylabel("Fare")

plt.suptitle("KNN：k を変えたときの決定境界の変化", fontsize=13)
plt.tight_layout()
plt.show()


k が小さいほど境界はどうなりましたか？これは過学習・汎化のどちらに近い状態ですか？

---

## 6. 8モデル一斉実装・精度比較

### モデルの定義


In [ ]:
models = {
    "Logistic Regression" : LogisticRegression(max_iter=1000),
    "Decision Tree"       : DecisionTreeClassifier(max_depth=5, random_state=42),
    "Random Forest"       : RandomForestClassifier(n_estimators=100, random_state=42, n_jobs=-1),
    "Gradient Boosting"   : GradientBoostingClassifier(n_estimators=100, random_state=42),
    "KNN (k=5)"           : KNeighborsClassifier(n_neighbors=5),
    "SVM (RBF)"           : SVC(kernel='rbf'),
    "Naive Bayes"         : GaussianNB(),
    "Neural Net"          : MLPClassifier(hidden_layer_sizes=(100, 50), max_iter=1000, random_state=42),
}


### 全モデルの学習と精度比較


In [ ]:
results = {}

for name, model in models.items():
    model.fit(X_train, y_train)
    acc = accuracy_score(y_test, model.predict(X_test))
    results[name] = acc
    print(f"{name:25s}: {acc:.4f}  ({acc*100:.1f}%)")

best_name = max(results, key=results.get)
worst_name = min(results, key=results.get)
print(f"\n🏆 最高精度: {best_name}  →  {results[best_name]:.4f}")
print(f"📉 最低精度: {worst_name}  →  {results[worst_name]:.4f}")
print(f"差: {(results[best_name] - results[worst_name])*100:.1f} ポイント")


### 📝 演習 6-1

上のコードを実行して、8モデルの精度を確認してください。どのモデルが最も高い精度でしたか？予想と一致しましたか？

### 精度を棒グラフで比較する


In [ ]:
result_series = pd.Series(results).sort_values(ascending=True)

colors = ['#59a14f' if name == result_series.idxmax() else '#4e79a7'
          for name in result_series.index]

plt.figure(figsize=(10, 6))
bars = plt.barh(result_series.index, result_series.values, color=colors)
plt.axvline(x=result_series.max(), color='green', linestyle='--', alpha=0.6,
            label=f"最高精度: {result_series.max():.4f}")
plt.xlabel("正解率 (Accuracy)", fontsize=12)
plt.title("8分類器の精度比較（全特徴量使用）", fontsize=14)
plt.xlim(0.6, 1.0)

# 各バーに精度の数値を表示
for bar, val in zip(bars, result_series.values):
    plt.text(val + 0.002, bar.get_y() + bar.get_height()/2,
             f'{val:.4f}', va='center', fontsize=9)

plt.legend()
plt.tight_layout()
plt.show()


### 特徴量の重要度を確認する（Random Forest）

Random Forestには、どの特徴量が予測に貢献しているかを示す**特徴量重要度（Feature Importance）** が備わっています。


In [ ]:
rf_model = models["Random Forest"]
importances = pd.Series(rf_model.feature_importances_, index=X.columns)
importances = importances.sort_values(ascending=True)

plt.figure(figsize=(8, 5))
importances.plot.barh(color='steelblue')
plt.title("Random Forest：特徴量の重要度", fontsize=14)
plt.xlabel("重要度（値が大きいほど予測への貢献が大きい）")
plt.tight_layout()
plt.show()

print("特徴量重要度（降順）:")
print(importances.sort_values(ascending=False).round(4))


### 📝 演習 6-2（DecisionTree の max_depth）

`max_depth` を変えて過学習を観察してください。


In [ ]:
for depth in [2, 5, 10, None]:
    dt = DecisionTreeClassifier(max_depth=depth, random_state=42)
    dt.fit(X_train, y_train)
    train_acc = accuracy_score(y_train, dt.predict(X_train))
    test_acc  = accuracy_score(y_test,  dt.predict(X_test))
    label = str(depth) if depth else "None（制限なし）"
    print(f"max_depth={label:12s}  訓練: {train_acc:.4f}  テスト: {test_acc:.4f}  差: {(train_acc-test_acc):.4f}")


`max_depth=None` のとき、訓練精度とテスト精度の差が最も大きくなっています。これが過学習の典型的なパターンです。`max_depth` はいくつのとき最もテスト精度が高かったですか？

### 📝 演習 6-3（KNN の n_neighbors）

`n_neighbors` を変えて精度がどう変わるか確認してください。


In [ ]:
k_results = {}
for k in [1, 3, 5, 10, 15, 20]:
    knn = KNeighborsClassifier(n_neighbors=k)
    knn.fit(X_train, y_train)
    train_acc = accuracy_score(y_train, knn.predict(X_train))
    test_acc  = accuracy_score(y_test,  knn.predict(X_test))
    k_results[k] = test_acc
    print(f"k={k:3d}  訓練: {train_acc:.4f}  テスト: {test_acc:.4f}")

best_k = max(k_results, key=k_results.get)
print(f"\nテスト精度が最も高いk: {best_k}")


k=1 のとき訓練精度はなぜ1.0000（100%）になるのか、説明できますか？

### 📝 演習 6-4（RandomForest の n_estimators）

木の本数（`n_estimators`）を変えて精度がどう変わるか確認してください。


In [ ]:
for n in [10, 50, 100, 200, 500]:
    rf = RandomForestClassifier(n_estimators=n, random_state=42, n_jobs=-1)
    rf.fit(X_train, y_train)
    acc = accuracy_score(y_test, rf.predict(X_test))
    print(f"n_estimators={n:4d}  テスト精度: {acc:.4f}")


木の数が増えると精度は単調に向上しますか？どこかで頭打ちになりますか？

### 📝 演習 6-5（応用：family_size の追加）

演習 3-5 で作った `family_size` 列を `X` に追加して再学習し、精度が向上するか確認してください。


In [ ]:
X_new = df_ml.drop('survived', axis=1).copy()
X_new['family_size'] = X_new['sibsp'] + X_new['parch'] + 1
X_new['is_alone'] = (X_new['family_size'] == 1).astype(int)  # 1人旅か否か

X_train2, X_test2, y_train2, y_test2 = train_test_split(
    X_new, y, test_size=0.2, random_state=42, stratify=y
)

print("--- 新特徴量追加後の精度 ---")
for name, model in models.items():
    model2 = type(model)()  # 同じ種類のモデルを新規作成
    model2.fit(X_train2, y_train2)
    acc2 = accuracy_score(y_test2, model2.predict(X_test2))
    diff = acc2 - results[name]
    sign = "↑" if diff > 0 else "↓" if diff < 0 else "→"
    print(f"{name:25s}: {acc2:.4f}  ({sign}{abs(diff)*100:.1f}pt)")


---

## 7. 最良モデルの混同行列

正解率だけでは「どこを間違えているか」が分かりません。**混同行列（Confusion Matrix）** でモデルの間違いのパターンを詳しく確認します。

### 混同行列の読み方

```
                   予測: 死亡(0)    予測: 生存(1)
実際: 死亡(0)      TN（正解）        FP（誤検知）
実際: 生存(1)      FN（見逃し）      TP（正解）
```

| 略語                     | 意味                   | タイタニックの例                             |
| ------------------------ | ---------------------- | -------------------------------------------- |
| **TP**（True Positive）  | 生存を生存と正しく予測 | 「生きた人を正しく把握」                     |
| **TN**（True Negative）  | 死亡を死亡と正しく予測 | 「亡くなった人を正しく把握」                 |
| **FP**（False Positive） | 死亡を生存と誤予測     | 「亡くなった人を生存と誤判断」               |
| **FN**（False Negative） | 生存を死亡と誤予測     | 「生存者を死亡と見落とし」← 救助場面では最悪 |

### 精度指標の整理

```
正解率（Accuracy）  = (TP + TN) / 全体       ← 全体の正解率
適合率（Precision） = TP / (TP + FP)          ← 「生存予測」の信頼度
再現率（Recall）    = TP / (TP + FN)          ← 生存者の発見率
F1スコア            = 2 × Precision × Recall / (Precision + Recall)
```


In [ ]:
best_name = max(results, key=results.get)
y_pred_best = models[best_name].predict(X_test)
cm = confusion_matrix(y_test, y_pred_best)

plt.figure(figsize=(6, 5))
disp = ConfusionMatrixDisplay(confusion_matrix=cm,
                               display_labels=["死亡(0)", "生存(1)"])
disp.plot(ax=plt.gca(), cmap="Blues", colorbar=False)
plt.title(f"混同行列：{best_name}\n正解率: {results[best_name]:.4f}", fontsize=13)
plt.tight_layout()
plt.show()

TN, FP, FN, TP = cm[0,0], cm[0,1], cm[1,0], cm[1,1]
print(f"TN（死亡→死亡と予測）: {TN} 件")
print(f"TP（生存→生存と予測）: {TP} 件")
print(f"FP（死亡→生存と誤予測）: {FP} 件")
print(f"FN（生存→死亡と誤予測）: {FN} 件")
precision = TP / (TP + FP)
recall    = TP / (TP + FN)
f1        = 2 * precision * recall / (precision + recall)
print(f"\n適合率（Precision）: {precision:.4f}")
print(f"再現率（Recall）:     {recall:.4f}")
print(f"F1スコア:             {f1:.4f}")


### 📝 演習 7-1

混同行列の結果を見て、以下を答えてください。

1. このモデルは「生存者の見逃し（FN）」と「死亡者の誤検知（FP）」のどちらが多いですか？
2. タイタニックの実際の救助活動という文脈では、FNとFPのどちらが「より深刻な間違い」ですか？理由も書いてください。

### 📝 演習 7-2

精度が**最も低かったモデル**の混同行列も表示し、最良モデルと比較してください。  
TN・TP・FP・FNのどの部分で差が出ましたか？

### 📝 演習 7-3

`classification_report` を使って、最良モデルの詳細なレポートを表示してください。


In [ ]:
from sklearn.metrics import classification_report
print(f"=== {best_name} の詳細レポート ===")
print(classification_report(y_test, y_pred_best, target_names=["死亡(0)", "生存(1)"]))


`precision`・`recall`・`f1-score`・`support` それぞれの値を読み取り、何を意味するか説明してください。

### 📝 演習 7-4（応用）

全8モデルの混同行列を一度に並べて表示してください。


In [ ]:
fig, axes = plt.subplots(2, 4, figsize=(18, 9))
axes = axes.flatten()

for ax, (name, model) in zip(axes, models.items()):
    y_pred_i = model.predict(X_test)
    cm_i = confusion_matrix(y_test, y_pred_i)
    ConfusionMatrixDisplay(cm_i, display_labels=["死亡", "生存"]).plot(
        ax=ax, cmap="Blues", colorbar=False
    )
    ax.set_title(f"{name}\n精度: {results[name]:.4f}", fontsize=9)

plt.suptitle("全8モデルの混同行列", fontsize=13, fontweight='bold')
plt.tight_layout()
plt.show()


---

## 🏆 本日の総合課題

以下のステップをすべて実行し、考察をコードのコメントとして記録してください。


In [ ]:
# ステップ1：データ読み込みから train_test_split まで一通り実行する
# （欠損値の確認・補完・文字列変換・X と y の分離・stratify 付き分割）


In [ ]:
# ステップ2：生存率に関わるグラフを3つ以上描く
# （sns.histplot・sns.boxplot・sns.heatmap を少なくとも1回ずつ使う。引数を意識すること）


In [ ]:
# ステップ3：8モデルを学習・比較し、精度を棒グラフで表示する
# 最良モデルを特定し、その混同行列と classification_report を表示する


In [ ]:
# ステップ4：以下をコメントで書く（各項目2〜3文）
# - 最良モデルはどれか、正解率は何%か
# - 特徴量重要度から「生存を左右した要因」を考察する
# - FN と FP のどちらが多く、それは問題かどうか
# - 2特徴量の決定境界グラフから気づいたことと全特徴量での精度の違い


In [ ]:
# ステップ5（応用）：以下のどちらかに挑戦する
#
# A) 特徴量エンジニアリング
#    family_size・is_alone・fare_per_person（運賃÷家族人数）などを追加して
#    全モデルを再学習し、精度が最も上がったモデルと特徴量を報告する
#
# B) ハイパーパラメータ自動探索（GridSearchCV）
#    以下のコードで Random Forest の最適なパラメータを自動探索する
#
# from sklearn.model_selection import GridSearchCV
# param_grid = {
#     'n_estimators': [50, 100, 200],
#     'max_depth': [3, 5, 10, None],
#     'min_samples_leaf': [1, 2, 4]
# }
# gs = GridSearchCV(RandomForestClassifier(random_state=42), param_grid,
#                   cv=5, scoring='accuracy', n_jobs=-1)
# gs.fit(X_train, y_train)
# print("最適パラメータ:", gs.best_params_)
# print("CV精度:", round(gs.best_score_, 4))
# print("テスト精度:", round(accuracy_score(y_test, gs.predict(X_test)), 4))
